In [ ]:
#analyzing congress right now 
import pandas as pd
import plotly.io as pio
import plotly.express as px
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize

pio.renderers.default = "browser"

# reading
votes = pd.read_csv('data/HS119/HS119_votes.csv')
members = pd.read_csv('data/HS119/HS119_members.csv')
roll_calls = pd.read_csv('data/HS119   /HS119_rollcalls.csv')

house_votes = votes[votes["chamber"] == "House"]
senate_votes = votes[votes["chamber"] == "Senate"]

house_members = members[members["chamber"] == "House"]
senate_members = members[members["chamber"] == "Senate"]

# mapping
def convert_vote(code):
    if code in [1, 2, 3]:
        return 1;
    elif code in [4, 5, 6]:
        return -1;
    else:
        return 0;

house_votes["votes_numeric"] = votes["cast_code"].apply(convert_vote)
senate_votes["votes_numeric"] = votes["cast_code"].apply(convert_vote)

house_matrix = house_votes.pivot_table(index="icpsr", columns="rollnumber", values="votes_numeric", fill_value=0)
senate_matrix = senate_votes.pivot_table(index="icpsr", columns="rollnumber", values="votes_numeric", fill_value=0)

# Keep only legislators with at least 100 cast votes (non-zero entries).
house_matrix = house_matrix[(house_matrix != 0).sum(axis=1) >= 100]
senate_matrix = senate_matrix[(senate_matrix != 0).sum(axis=1) >= 100]


# L2-normalize each legislator vector (row) before PCA
house_matrix_norm = pd.DataFrame(
    normalize(house_matrix, norm="l2", axis=1),
    index=house_matrix.index,
    columns=house_matrix.columns,
 )
senate_matrix_norm = pd.DataFrame(
    normalize(senate_matrix, norm="l2", axis=1),
    index=senate_matrix.index,
    columns=senate_matrix.columns,
 )

# PCA

pca = PCA(n_components=2)
coords_house = pca.fit_transform(house_matrix_norm)
coords_senate = pca.fit_transform(senate_matrix_norm)

# plot house
df_house = pd.DataFrame(coords_house, columns=["PC1", "PC2"])
df_house["icpsr"] = house_matrix_norm.index

df_house = df_house.merge(
    house_members[["icpsr", "party_code", "bioname"]],
    on="icpsr"
)


fig = px.scatter(
    df_house,
    title="House",
    x="PC1",
    y="PC2",
    color="party_code",
    hover_name="bioname"
)

fig.show()

# plot senate
df_senate = pd.DataFrame(coords_senate, columns=["PC1", "PC2"])
df_senate["icpsr"] = senate_matrix_norm.index

df_senate = df_senate.merge(
    senate_members[["icpsr", "party_code", "bioname"]],
    on="icpsr"
)


fig = px.scatter(
    df_senate,
    title="Senate",
    x="PC1",
    y="PC2",
    color="party_code",
    hover_name="bioname"
)

fig.show()

In [1]:
# all congress polarization
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
import subprocess
from pathlib import Path

# reading
votes = pd.read_csv('data/all/HSall_votes.csv')
members = pd.read_csv('data/all/HSall_members.csv')

def convert_vote(code):
    if code in [1, 2, 3]:
        return 1
    elif code in [4, 5, 6]:
        return -1
    elif code in [7, 8]:
        return 0
    else:
        return np.nan
    
votes["vote_numeric"] = votes["cast_code"].apply(convert_vote)

polarization_results = []

for congress in sorted(votes["congress"].unique()):
    v = votes[votes["congress"] == congress]

    matrix = v.pivot_table(
        index="icpsr",
        columns="rollnumber",
        values="vote_numeric"
    )

    # Require at least 100 votes
    vote_counts = matrix.notna().sum(axis=1)
    matrix = matrix[vote_counts >= 100]

    if matrix.shape[0] < 10 or matrix.shape[1] < 5:
        continue

    M = matrix.fillna(0).values

    # Row normalize
    norms = np.linalg.norm(M, axis=1, keepdims=True)
    M_norm = M / np.where(norms == 0, 1, norms)

    pca = PCA(n_components=3)
    coords = pca.fit_transform(M_norm)

    pc1_variance = np.var(coords[:, 0])
    explained_pc1 = pca.explained_variance_ratio_[0]

    polarization_results.append({
        "congress": congress,
        "pc1_variance_polarization": pc1_variance,
        "pc1_explained_variance": explained_pc1,
        "num_legislators": matrix.shape[0],
        "num_votes": matrix.shape[1]
    })

polarization_df = pd.DataFrame(polarization_results)
print(polarization_df.head())

import plotly.express as px

fig = px.line(
    polarization_df,
    x="congress",
    y="pc1_variance_polarization",
    markers=True,
    title="Party-Independent Polarization Over Time",
    labels={
        "congress": "Congress",
        "pc1_variance_polarization": "PC1 Variance"
    }
)

output_path = Path("polarization.html")
fig.write_html(output_path, include_plotlyjs="cdn", auto_open=False)
subprocess.Popen(["open", "-a", "Google Chrome", str(output_path)])




   congress  pc1_variance_polarization  pc1_explained_variance  \
0         1                   0.243826                0.277607   
1         5                   0.422047                0.448816   
2         6                   0.530714                0.608192   
3         7                   0.448568                0.523501   
4         8                   0.252008                0.283633   

   num_legislators  num_votes  
0               30        109  
1              114        202  
2               19        120  
3               73        142  
4              110        150  


<Popen: returncode: None args: ['open', '-a', 'Google Chrome', 'polarization...>

In [2]:
# modern polarization, starting in the 1970s. difference between party centroids 
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA

MODERN_START = 93   # change this to 91, 92, 93, etc.
MIN_VOTES = 100

def convert_vote(code):
    if code in [1, 2, 3]:
        return 1
    elif code in [4, 5, 6]:
        return -1
    elif code in [7, 8]:
        return 0
    else:
        return np.nan

votes["vote_numeric"] = votes["cast_code"].apply(convert_vote)

modern_results = []

for congress in sorted(votes["congress"].unique()):
    if congress < MODERN_START:
        continue

    v = votes[votes["congress"] == congress]

    matrix = v.pivot_table(
        index="icpsr",
        columns="rollnumber",
        values="vote_numeric"
    )

    vote_counts = matrix.notna().sum(axis=1)
    matrix = matrix[vote_counts >= MIN_VOTES]

    if matrix.shape[0] < 10 or matrix.shape[1] < 5:
        continue

    M = matrix.fillna(0).values

    norms = np.linalg.norm(M, axis=1, keepdims=True)
    M_norm = M / np.where(norms == 0, 1, norms)

    pca = PCA(n_components=2)
    coords = pca.fit_transform(M_norm)

    df = pd.DataFrame(coords, columns=["PC1", "PC2"])
    df["icpsr"] = matrix.index

    df = df.merge(
        members[["icpsr", "party_code", "bioname"]],
        on="icpsr",
        how="left"
    )

    # Voteview: 100 = Democrat, 200 = Republican
    df = df[df["party_code"].isin([100, 200])]

    if df["party_code"].nunique() < 2:
        continue

    dem_center = df[df["party_code"] == 100][["PC1", "PC2"]].mean()
    rep_center = df[df["party_code"] == 200][["PC1", "PC2"]].mean()

    distance = np.linalg.norm(rep_center - dem_center)

    modern_results.append({
        "congress": congress,
        "party_centroid_distance": distance,
        "dem_pc1_mean": dem_center["PC1"],
        "rep_pc1_mean": rep_center["PC1"],
        "dem_pc2_mean": dem_center["PC2"],
        "rep_pc2_mean": rep_center["PC2"],
        "num_legislators": len(df),
        "num_votes": matrix.shape[1]
    })

modern_polarization = pd.DataFrame(modern_results)
print(modern_polarization.head())

import plotly.express as px

fig = px.line(
    modern_polarization,
    x="congress",
    y="party_centroid_distance",
    markers=True,
    title=f"Democrat-Republican Polarization Since Congress {MODERN_START}",
    labels={
        "congress": "Congress",
        "party_centroid_distance": "Distance Between Party Centers"
    }
)

fig.show()

   congress  party_centroid_distance  dem_pc1_mean  rep_pc1_mean  \
0        93                 0.458910     -0.197581      0.258244   
1        94                 0.536054     -0.175470      0.358367   
2        95                 0.512466     -0.177585      0.332187   
3        96                 0.612728     -0.243457      0.369038   
4        97                 0.571110     -0.275686      0.285594   

   dem_pc2_mean  rep_pc2_mean  num_legislators  num_votes  
0      0.047306     -0.005816             5469       1138  
1      0.033744     -0.014965             5541       1311  
2      0.038648     -0.013829             5492       1540  
3      0.034883      0.017985             5493       1276  
4     -0.024321      0.081184             5495        966  


In [2]:
# member ideology trajectories by congress
from pathlib import Path
import json

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize

DATA_DIR = Path("data/all")
OUTPUT_DIR = Path("data/derived")
SITE_DATA_DIR = Path("site-data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SITE_DATA_DIR.mkdir(parents=True, exist_ok=True)

VALID_CHAMBERS = {"House", "Senate"}
DEMOCRAT_CODES = {100, 328}
REPUBLICAN_CODES = {200}

def convert_vote(code):
    if code in (1, 2, 3):
        return 1
    if code in (4, 5, 6):
        return -1
    return 0

def build_year_labels(rollcalls):
    labels = {}
    for congress, group in rollcalls.groupby("congress"):
        dates = pd.to_datetime(group["date"], errors="coerce").dropna()
        if dates.empty:
            labels[int(congress)] = str(int(congress))
            continue
        start_year = int(dates.min().year)
        end_year = int(dates.max().year)
        labels[int(congress)] = str(start_year) if start_year == end_year else f"{start_year}-{end_year}"
    return labels

def align_ideology_axis(df):
    aligned = df.copy()
    dem_mask = aligned["party_code"].isin(DEMOCRAT_CODES)
    rep_mask = aligned["party_code"].isin(REPUBLICAN_CODES)

    if dem_mask.any() and rep_mask.any():
        dem_mean = aligned.loc[dem_mask, "pc1"].mean()
        rep_mean = aligned.loc[rep_mask, "pc1"].mean()
        if rep_mean < dem_mean:
            aligned["pc1"] *= -1
            aligned["pc2"] *= -1

    ideology_std = aligned["pc1"].std(ddof=0)
    ideology_std = float(ideology_std) if pd.notna(ideology_std) and ideology_std > 0 else 1.0
    ideology_mean = float(aligned["pc1"].mean())
    aligned["display_ideology_score"] = (aligned["pc1"] - ideology_mean) / ideology_std

    dem_mask = aligned["party_code"].isin(DEMOCRAT_CODES)
    rep_mask = aligned["party_code"].isin(REPUBLICAN_CODES)
    if dem_mask.any() and rep_mask.any():
        dem_mean = float(aligned.loc[dem_mask, "pc1"].mean())
        rep_mean = float(aligned.loc[rep_mask, "pc1"].mean())
        gap = rep_mean - dem_mean
        midpoint = (rep_mean + dem_mean) / 2.0
        aligned["party_gap_score"] = (aligned["pc1"] - midpoint) / gap if abs(gap) > 1e-12 else np.nan
    else:
        aligned["party_gap_score"] = np.nan

    return aligned

def compute_member_ideology(votes, members, year_labels, chamber, min_votes=100, min_legislators=10, min_rollcalls=5):
    chamber_votes = votes[votes["chamber"] == chamber].copy()
    chamber_members = members[members["chamber"] == chamber].copy()
    chamber_votes["vote_numeric"] = chamber_votes["cast_code"].apply(convert_vote)

    result_frames = []
    meta_rows = []

    for congress, congress_votes in chamber_votes.groupby("congress", sort=True):
        congress_members = chamber_members[chamber_members["congress"] == congress]
        if congress_members.empty:
            continue

        matrix = congress_votes.pivot_table(index="icpsr", columns="rollnumber", values="vote_numeric", fill_value=0)
        if matrix.empty:
            continue

        vote_counts = (matrix != 0).sum(axis=1)
        matrix = matrix[vote_counts >= min_votes]
        if matrix.shape[0] < min_legislators or matrix.shape[1] < min_rollcalls:
            continue

        coords = PCA(n_components=2).fit_transform(normalize(matrix, norm="l2", axis=1))
        ideology_df = pd.DataFrame({
            "icpsr": matrix.index.astype(int),
            "pc1": coords[:, 0],
            "pc2": coords[:, 1],
            "cast_vote_count": vote_counts.loc[matrix.index].to_numpy(),
        })

        ideology_df = ideology_df.merge(
            congress_members[["icpsr", "party_code", "bioname", "state_abbrev", "district_code", "bioguide_id"]],
            on="icpsr",
            how="left",
        )
        ideology_df = align_ideology_axis(ideology_df)
        ideology_df["congress"] = int(congress)
        ideology_df["year_label"] = year_labels.get(int(congress), str(int(congress)))
        ideology_df["chamber"] = chamber
        result_frames.append(ideology_df)

        meta_rows.append({
            "congress": int(congress),
            "year_label": year_labels.get(int(congress), str(int(congress))),
            "chamber": chamber,
            "num_legislators": int(matrix.shape[0]),
            "num_rollcalls": int(matrix.shape[1]),
        })

    return pd.concat(result_frames, ignore_index=True), pd.DataFrame(meta_rows)

def build_member_summary(records):
    summary_rows = []
    for icpsr, group in records.groupby("icpsr"):
        ordered = group.sort_values(["congress", "chamber"]).reset_index(drop=True)
        scores = ordered["display_ideology_score"].to_numpy()
        deltas = np.diff(scores) if len(scores) > 1 else np.array([])
        summary_rows.append({
            "icpsr": int(icpsr),
            "bioname": ordered.loc[0, "bioname"],
            "appearances": int(len(ordered)),
            "first_congress": int(ordered.loc[0, "congress"]),
            "last_congress": int(ordered.loc[len(ordered) - 1, "congress"]),
            "net_shift": float(scores[-1] - scores[0]),
            "cumulative_shift": float(np.abs(deltas).sum()) if len(deltas) else 0.0,
        })
    return pd.DataFrame(summary_rows).sort_values(["appearances", "last_congress", "bioname"], ascending=[False, False, True])

def write_json(path, rows):
    path.write_text(json.dumps(rows, indent=2), encoding="utf-8")

votes = pd.read_csv(DATA_DIR / "HSall_votes.csv", usecols=["congress", "chamber", "rollnumber", "icpsr", "cast_code"], low_memory=False)
members = pd.read_csv(DATA_DIR / "HSall_members.csv", usecols=["congress", "chamber", "icpsr", "party_code", "bioname", "state_abbrev", "district_code", "bioguide_id"], low_memory=False)
rollcalls = pd.read_csv(DATA_DIR / "HSall_rollcalls.csv", usecols=["congress", "date"], low_memory=False)

members = members[members["chamber"].isin(VALID_CHAMBERS)].copy()
year_labels = build_year_labels(rollcalls)

ideology_frames = []
meta_frames = []
for chamber in ["House", "Senate"]:
    records, meta = compute_member_ideology(votes, members, year_labels, chamber)
    ideology_frames.append(records)
    meta_frames.append(meta)

ideology_records = pd.concat(ideology_frames, ignore_index=True).sort_values(["bioname", "congress", "chamber"])
congress_meta = pd.concat(meta_frames, ignore_index=True).sort_values(["congress", "chamber"])
summary = build_member_summary(ideology_records)

ideology_records.to_csv(OUTPUT_DIR / "member_ideology_records.csv", index=False)
summary.to_csv(OUTPUT_DIR / "member_ideology_summary.csv", index=False)
congress_meta.to_csv(OUTPUT_DIR / "congress_ideology_metadata.csv", index=False)
write_json(OUTPUT_DIR / "member_ideology_records.json", ideology_records.replace({np.nan: None}).to_dict(orient="records"))
write_json(SITE_DATA_DIR / "member_ideology_records.json", ideology_records.replace({np.nan: None}).to_dict(orient="records"))
write_json(OUTPUT_DIR / "member_ideology_summary.json", summary.replace({np.nan: None}).to_dict(orient="records"))

summary[["bioname", "appearances", "first_congress", "last_congress", "net_shift"]].head(20)


,bioname,appearances,first_congress,last_congress,net_shift
2207,"DINGELL, John David, Jr.",29,85,113,0.131700
1161,"BYRD, Robert Carlyle",29,83,111,0.899327
3566,"HAYDEN, Carl Trumbull",28,62,90,0.228954
8894,"CONYERS, John, Jr.",27,89,115,0.028835
4038,"INOUYE, Daniel Ken",27,86,112,-0.190791
9310,"GRASSLEY, Charles Ernest",26,94,119,-0.244699
9420,"MARKEY, Edward John",26,95,119,0.063692
8410,"WHITTEN, Jamie Lloyd",26,78,103,-0.146530
9263,"YOUNG, Donald Edwin",25,93,117,-0.270292
9640,"WYDEN, Ronald Lee",24,97,119,0.155718


In [3]:
# topic ideology analysis for current legislators by chamber
from pathlib import Path
from IPython.display import display, Markdown
import json
import re

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.decomposition import PCA

DATA_DIR = Path("data/all")
SITE_DATA_DIR = Path("site-data")
OUTPUT_DIR = Path("data/derived")
SITE_DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CURRENT_MEMBERS_PATH = Path("site-data/HS119/HS119_members.csv")
TOPIC_SCORE_NAMES = {
    "Defense / Military": "defense_score",
    "Economy / Budget / Taxes": "economy_score",
    "Healthcare": "healthcare_score",
    "Social Policy": "social_policy_score",
    "Foreign Policy": "foreign_policy_score",
    "Other": "other_score",
}
MIN_MEMBER_TOPIC_VOTES = 20
MIN_ROLLCALL_PARTICIPANTS = 20
MIN_TOPIC_ROLLCALLS = 8
VALID_CHAMBERS = ["House", "Senate"]


def convert_vote(code):
    code = pd.to_numeric(code, errors="coerce")
    if code in (1, 2, 3):
        return 1.0
    if code in (4, 5, 6):
        return -1.0
    if code == 7:
        return 0.0
    return np.nan


def party_label(party_code):
    if party_code in (100, 328):
        return "Democrat"
    if party_code == 200:
        return "Republican"
    return "Other"


def score_column_name(topic):
    if topic in TOPIC_SCORE_NAMES:
        return TOPIC_SCORE_NAMES[topic]
    slug = re.sub(r"[^a-z0-9]+", "_", topic.lower()).strip("_")
    return f"{slug}_score"


def load_current_members():
    usecols = ["chamber", "icpsr", "party_code", "bioname", "bioguide_id", "state_abbrev", "district_code", "nokken_poole_dim1"]
    current = pd.read_csv(CURRENT_MEMBERS_PATH, usecols=usecols, low_memory=False)
    current = current[current["chamber"].isin(VALID_CHAMBERS)].copy()
    current["icpsr"] = current["icpsr"].astype(int)
    current["party"] = current["party_code"].apply(party_label)
    return current.drop_duplicates(["chamber", "icpsr"])


def load_topic_rollcalls():
    usecols = ["congress", "chamber", "rollnumber", "topic_category", "date"]
    rollcalls = pd.read_csv(DATA_DIR / "HSall_rollcalls_categorized.csv", usecols=usecols, low_memory=False)
    rollcalls = rollcalls[rollcalls["chamber"].isin(VALID_CHAMBERS)].copy()
    rollcalls = rollcalls[rollcalls["topic_category"].notna()]
    rollcalls["congress"] = rollcalls["congress"].astype(int)
    rollcalls["rollnumber"] = rollcalls["rollnumber"].astype(int)
    return rollcalls


def load_current_votes(active_icpsr, chamber):
    usecols = ["congress", "chamber", "rollnumber", "icpsr", "cast_code"]
    chunks = []
    for chunk in pd.read_csv(DATA_DIR / "HSall_votes.csv", usecols=usecols, chunksize=250_000, low_memory=False):
        chunk = chunk[(chunk["chamber"] == chamber) & (chunk["icpsr"].isin(active_icpsr))].copy()
        if chunk.empty:
            continue
        chunk["congress"] = chunk["congress"].astype(int)
        chunk["rollnumber"] = chunk["rollnumber"].astype(int)
        chunk["icpsr"] = chunk["icpsr"].astype(int)
        chunk["vote_numeric"] = chunk["cast_code"].apply(convert_vote)
        chunks.append(chunk[["congress", "chamber", "rollnumber", "icpsr", "vote_numeric"]])
    return pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame(columns=["congress", "chamber", "rollnumber", "icpsr", "vote_numeric"])


def build_topic_matrix(topic_votes, member_index):
    matrix = topic_votes.pivot_table(index="icpsr", columns=["congress", "rollnumber"], values="vote_numeric", aggfunc="first")
    matrix = matrix.reindex(member_index)
    matrix = matrix.loc[matrix.notna().sum(axis=1) >= MIN_MEMBER_TOPIC_VOTES]
    matrix = matrix.loc[:, matrix.notna().sum(axis=0) >= MIN_ROLLCALL_PARTICIPANTS]
    if matrix.shape[0] < 3 or matrix.shape[1] < MIN_TOPIC_ROLLCALLS:
        return None
    return matrix


def align_pc1(scores, meta):
    aligned = scores.copy()
    rep = aligned[meta["party_code"] == 200]
    dem = aligned[meta["party_code"].isin([100, 328])]
    if len(rep) and len(dem) and rep.mean() < dem.mean():
        aligned = -aligned
    elif meta["nokken_poole_dim1"].notna().sum() >= 3:
        mask = meta["nokken_poole_dim1"].notna()
        corr = np.corrcoef(aligned[mask], meta.loc[mask, "nokken_poole_dim1"])[0, 1]
        if np.isfinite(corr) and corr < 0:
            aligned = -aligned
    return aligned


def topic_pc1_scores(matrix, member_meta):
    centered = matrix.sub(matrix.mean(axis=0, skipna=True), axis=1)
    filled = centered.fillna(0.0)
    values = filled.to_numpy(dtype=float)
    norms = np.linalg.norm(values, axis=1, keepdims=True)
    normalized = values / np.where(norms == 0, 1, norms)
    pca = PCA(n_components=1)
    raw_scores = pd.Series(pca.fit_transform(normalized).ravel(), index=filled.index)
    aligned = align_pc1(raw_scores, member_meta.loc[filled.index])
    scale = aligned.std(ddof=0)
    z_scores = (aligned - aligned.mean()) / (scale if scale else 1.0)
    return pd.DataFrame({
        "icpsr": filled.index,
        "pc1_score": aligned.to_numpy(),
        "topic_score": z_scores.to_numpy(),
        "vote_count": matrix.notna().sum(axis=1).to_numpy(),
        "explained_variance_ratio": float(pca.explained_variance_ratio_[0]),
    })


def cosine_similarity_with_missing(score_frame, min_overlap=25):
    topics = list(score_frame.columns)
    similarity = pd.DataFrame(np.eye(len(topics)), index=topics, columns=topics, dtype=float)
    for i, left in enumerate(topics):
        x_all = score_frame[left].to_numpy(dtype=float)
        for j in range(i + 1, len(topics)):
            right = topics[j]
            y_all = score_frame[right].to_numpy(dtype=float)
            mask = np.isfinite(x_all) & np.isfinite(y_all)
            if mask.sum() < min_overlap:
                value = np.nan
            else:
                x = x_all[mask]
                y = y_all[mask]
                denom = np.linalg.norm(x) * np.linalg.norm(y)
                value = np.nan if denom == 0 else float(np.dot(x, y) / denom)
            similarity.loc[left, right] = value
            similarity.loc[right, left] = value
    return similarity


def build_chamber_topic_analysis(current_members, topic_rollcalls, chamber):
    chamber_members = current_members[current_members["chamber"] == chamber].copy()
    active_icpsr = set(chamber_members["icpsr"])
    chamber_votes = load_current_votes(active_icpsr, chamber)
    topic_votes = chamber_votes.merge(topic_rollcalls[topic_rollcalls["chamber"] == chamber], on=["congress", "chamber", "rollnumber"], how="inner")

    member_meta = chamber_members.set_index("icpsr").sort_index()
    member_index = member_meta.index.tolist()
    topic_series = {}
    topic_meta_rows = []

    for topic, group in topic_votes.groupby("topic_category"):
        matrix = build_topic_matrix(group, member_index)
        if matrix is None:
            continue
        score_df = topic_pc1_scores(matrix, member_meta)
        score_series = pd.Series(np.nan, index=member_index, name=topic)
        score_series.loc[score_df["icpsr"]] = score_df["topic_score"].to_numpy()
        topic_series[topic] = score_series
        topic_meta_rows.append({
            "topic": topic,
            "score_column": score_column_name(topic),
            "num_rollcalls": int(matrix.shape[1]),
            "num_legislators": int(matrix.shape[0]),
            "mean_vote_count": float(score_df["vote_count"].mean()),
            "explained_variance_ratio": float(score_df["explained_variance_ratio"].iloc[0]),
        })

    topic_scores = pd.DataFrame(topic_series).reindex(member_index)
    topic_meta = pd.DataFrame(topic_meta_rows).sort_values(["num_rollcalls", "topic"], ascending=[False, True]).reset_index(drop=True)
    ordered_topics = topic_meta["topic"].tolist()
    topic_scores = topic_scores.reindex(columns=ordered_topics)
    similarity = cosine_similarity_with_missing(topic_scores)

    wide = chamber_members[["icpsr", "bioname", "party", "party_code", "state_abbrev", "district_code", "bioguide_id"]].copy()
    for topic in ordered_topics:
        wide[score_column_name(topic)] = topic_scores[topic].to_numpy()

    return wide.sort_values(["party", "bioname"]).reset_index(drop=True), topic_meta, similarity


def build_topic_payload(chamber, wide, topic_meta, similarity):
    topics = topic_meta["topic"].tolist()
    topic_columns = topic_meta["score_column"].tolist()
    legislators = []
    for _, row in wide.sort_values("bioname").iterrows():
        legislators.append({
            "icpsr": int(row["icpsr"]),
            "bioname": row["bioname"],
            "party": row["party"],
            "party_code": int(row["party_code"]),
            "state_abbrev": row["state_abbrev"],
            "district_code": None if pd.isna(row["district_code"]) else str(row["district_code"]),
            "topic_scores": {
                topic: (None if pd.isna(row[col]) else float(row[col]))
                for topic, col in zip(topics, topic_columns)
            },
        })
    return {
        "chamber": chamber,
        "topics": topic_meta.replace({np.nan: None}).to_dict(orient="records"),
        "similarity_topics": topics,
        "similarity_matrix": similarity.replace({np.nan: None}).values.tolist(),
        "legislators": legislators,
        "explanation": (
            "Each topic defines its own legislator-by-rollcall matrix. After column-centering observed votes, "
            "each legislator vector is L2-normalized so PCA emphasizes voting pattern rather than vote count. "
            "PC1 is the dominant topic-specific voting axis, and each legislator's topic score is their projection "
            "onto that axis, standardized within topic for comparability across topics."
        ),
    }


def make_similarity_heatmap(similarity, chamber):
    fig = px.imshow(
        similarity,
        color_continuous_scale="RdBu",
        zmin=-1,
        zmax=1,
        text_auto=".2f",
        aspect="auto",
        title=f"{chamber} topic similarity (cosine similarity of topic score vectors)",
    )
    fig.update_layout(coloraxis_colorbar_title="Cosine similarity")
    return fig


def make_legislator_profile(wide, topic_meta, legislator_name):
    row = wide.loc[wide["bioname"] == legislator_name].iloc[0]
    plot_df = pd.DataFrame({
        "topic": topic_meta["topic"],
        "score": [row[col] for col in topic_meta["score_column"]],
    }).dropna()
    fig = go.Figure(go.Bar(
        x=plot_df["topic"],
        y=plot_df["score"],
        marker_color="#f7c66f",
        customdata=np.column_stack([[row["bioname"]] * len(plot_df), [row["party"]] * len(plot_df)]),
        hovertemplate="<b>%{customdata[0]}</b><br>Party: %{customdata[1]}<br>Topic: %{x}<br>Topic score: %{y:.2f}<extra></extra>",
    ))
    fig.update_layout(
        title=f"{legislator_name} topic ideology profile",
        xaxis_title="Topic",
        yaxis_title="Standardized PC1 score",
        template="plotly_white",
    )
    return fig


def write_json(path, payload):
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")


current_members = load_current_members()
topic_rollcalls = load_topic_rollcalls()
results = {}
for chamber in VALID_CHAMBERS:
    wide_scores, topic_meta, similarity = build_chamber_topic_analysis(current_members, topic_rollcalls, chamber)
    results[chamber] = {
        "wide": wide_scores,
        "topic_meta": topic_meta,
        "similarity": similarity,
    }
    payload = build_topic_payload(chamber, wide_scores, topic_meta, similarity)
    wide_scores.to_csv(OUTPUT_DIR / f"topic_scores_{chamber.lower()}.csv", index=False)
    topic_meta.to_csv(OUTPUT_DIR / f"topic_metadata_{chamber.lower()}.csv", index=False)
    similarity.to_csv(OUTPUT_DIR / f"topic_similarity_{chamber.lower()}.csv")
    write_json(SITE_DATA_DIR / f"topic_analysis_{chamber.lower()}.json", payload)
    write_json(OUTPUT_DIR / f"topic_analysis_{chamber.lower()}.json", payload)

house_heatmap = make_similarity_heatmap(results["House"]["similarity"], "House")
senate_heatmap = make_similarity_heatmap(results["Senate"]["similarity"], "Senate")
default_house_member = results["House"]["wide"].sort_values("bioname").iloc[0]["bioname"]
default_senate_member = results["Senate"]["wide"].sort_values("bioname").iloc[0]["bioname"]
house_profile = make_legislator_profile(results["House"]["wide"], results["House"]["topic_meta"], default_house_member)
senate_profile = make_legislator_profile(results["Senate"]["wide"], results["Senate"]["topic_meta"], default_senate_member)

display(Markdown("### Topic Analysis Outputs"))
display(Markdown("Each topic creates a separate voting matrix. PCA finds the dominant topic-specific voting direction, and each legislator's score is their projection onto that direction after column-centering and row normalization."))
display(house_heatmap)
display(house_profile)
display(senate_heatmap)
display(senate_profile)

results["House"]["wide"].head()


### Topic Analysis Outputs

Each topic creates a separate voting matrix. PCA finds the dominant topic-specific voting direction, and each legislator's score is their projection onto that direction after column-centering and row normalization.

,icpsr,bioname,party,party_code,state_abbrev,district_code,bioguide_id,other_score,economy_score,defense_score,social_policy_score,healthcare_score,foreign_policy_score
0,21545,"ADAMS, Alma",Democrat,100,NC,12,A000370,1.002217,0.847319,0.910973,0.932543,0.997948,1.009062
1,21506,"AGUILAR, Peter Rey",Democrat,100,CA,33,A000371,0.982982,0.912704,0.977704,1.011698,0.952360,0.893266
2,22375,"AMO, Gabe",Democrat,100,RI,1,A000380,0.779205,0.794808,0.525589,0.670023,NaN,NaN
3,22500,"ANSARI, Yassamin",Democrat,100,AZ,3,A000381,-0.989182,-0.922625,-1.016058,-1.129309,-1.011523,-1.006898
4,22100,"AUCHINCLOSS, Jake",Democrat,100,MA,4,A000148,-1.104644,-1.200986,-1.084997,-1.308289,-1.146706,-0.927409
